In [ ]:
from collections import Counter, defaultdict
from itertools import pairwise
import json
import pandas as pd
import re
from difflib import get_close_matches
from numpy import nan

In [ ]:
data = pd.read_parquet("sentences_parsed.parquet")
data.head()

In [ ]:
data.shape

In [ ]:
targets = pd.read_csv("noun_phrases_sample-classified.csv", sep=';')
targets.head()

In [ ]:
targets.change.unique()

In [ ]:
targets_set = set(targets.noun_phrase.to_list())
targets_set

In [ ]:
len(targets_set)

In [ ]:
count = 0
years_per_target = defaultdict(set)
green_list = []
for i, row in data.iterrows():
    splitted = row.lemmas.split()
    pairs = set([' '.join(pair) for pair in pairwise(splitted)])
    for intersected in pairs.intersection(targets_set):
        green_list.append(
            {"target": intersected, "example": row.sentence, "year": row.year, "doc_id": row.doc_id}
                         )
        count += 1
        years_per_target[intersected].add(row.year)
print(count)
# print(years_per_target)

In [ ]:
targets_single = set()
targets_many = set()
count = 0
for target, year in years_per_target.items():
    if len(year) > 1:
        count += 1
        targets_many.add(target)
    else:
        for word in target.split(): # always 2 words
            targets_single.add(word)
count

In [ ]:
len(targets_single)

In [ ]:
Counter(targets[targets.noun_phrase.isin(targets_many)].change)

Create a file, to which the model will be applied

In [ ]:
seen = set()
lines = []
with open("green.jsonl", "w") as f:
    for line in green_list:
        if (line["target"], line["example"]) not in seen:
            if line["target"] in targets_many:
                f.write(json.dumps(line)+'\n')
                seen.add((line["target"], line["example"]))
                lines.append(line)

Parse model's predictions

In [ ]:
to_split = "<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n\nYou are a lexicographer familiar with providing concise definitions of word meanings.<|eot_id|><|start_header_id|>user<|end_header_id|>\n\n"

In [ ]:
pred_fn = "green-long-ours.txt.gz"

In [ ]:
import gzip

In [ ]:
with gzip.open(pred_fn, "rt") as ours:
    ours_pred = ours.read()
ours_instances = ours_pred.split(to_split)
print(len(ours_instances))

In [ ]:
target_pat = re.compile(f"Please provide a concise definition for the meaning of the word \"(.+)\" in the following sentence:")
to_split_prompt = "<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n\n"

In [ ]:
to_split_prompt = "<|eot_id|>assistant\n\n"

In [ ]:
data.iloc[79904].sentence

In [ ]:
sent = """The US Environmental Protection Agency for the regulation of environmental and employee health
Luxembourg, 27 October 2017 In addition, Philips utilises environmental management systems and safety programmes and is conducting its business in compliance with all applicable environmental laws and regulations, notably: \uf0b7 The US Environmental Protection Agency for the regulation of environmental and employee health and the US Occupational Health and Safety Assessment System; \uf0b7 The European Union Registration, Evaluation, Authorisation and Restriction of Chemicals; \uf0b7 The ISO 14001 environment management certification (82% of manufacturing sites certified in 2016) \uf0b7 The ISO 9001 quality management certification."""

In [ ]:
data.iloc[70696].sentence == sent

In [ ]:
import difflib
import sys

In [ ]:
from tqdm import tqdm

In [ ]:
annotated = pd.read_csv("t1_ours_annotated.tsv", sep='\t', encoding="cp1252")
annotated.head()

In [ ]:
"" in annotated.full_phrase.unique()

In [ ]:
results_our = {"target": [], "prediction": [], "example": [], "year": [], "document": [], "lemmas":[], 
              "full_phrase": [],
              }

for i, instance_our in tqdm(enumerate(ours_instances[1:])):
    prompt_our, pred_our = instance_our.split(to_split_prompt)
    
    target_our = re.match(target_pat, prompt_our).group(1)
    sp = target_our.split(' ')
    if (target_our.lower() not in {"it", "which", "that", "this"}) and (not (len(sp)==2)) and (not (sp[0].lower() in {"the", "an"})):
        example = prompt_our.split(": ", maxsplit=1)[1]
        fphs = annotated[annotated.examples.str.contains(re.escape(example))].full_phrase.to_list()
        example = example.replace("\\uf0b7", "\uf0b7").replace(
            "\\uf0a7", "\uf0a7").replace(" ? ", " − ")
        if not fphs:
            print(example)
        full_phrase = set([str(x) for x in fphs if x is not nan])
        
        sentence = data[data.sentence==example]
        try:
            year = sentence.year.values[0]
        except IndexError as e:
            print(example)
            output_list = [li for li in difflib.ndiff(example, data.iloc[70696].sentence) if li[0] != ' ']
            print(output_list)
            raise e
        document = sentence.doc_id.values[0]
        results_our["full_phrase"].append(', '.join(full_phrase))
        results_our["target"].append(target_our)
        results_our["prediction"].append(pred_our.rstrip())
        results_our["example"].append(example)
        results_our["year"].append(year)
        results_our["document"].append(document)
        lemmas = sentence.lemmas.values[0]
        results_our["lemmas"].append(lemmas)
results = pd.DataFrame(results_our)
results.head()

In [ ]:
results[results.full_phrase==""]

In [ ]:
results.shape

In [ ]:
results.full_phrase.unique()

In [ ]:
not_review = results[~results.full_phrase.str.contains("review")]

In [ ]:
to_preserve = set()
for full_phrase, group in not_review.groupby("full_phrase"):
    print(full_phrase)
    targets = group.target.unique()
    print(targets)
    target = get_close_matches(full_phrase, targets, n=1)
    print(target)
    if len(target) == 1: # always 1 if not empty
        to_preserve.add(target[0]) 
    elif len(target) == 0:
        to_preserve.update(set(targets))
    else:
        raise Exception
    print("----------")

In [ ]:
to_preserve

In [ ]:
results_review = results[results.full_phrase.str.contains("review")]
results_review

In [ ]:
results_nice = results[results.target.isin(to_preserve) & ~results.full_phrase.str.contains("review")]
results_nice

In [ ]:
results_nice.target.unique()

In [ ]:
results.to_csv("green-long-cleaned.tsv", sep='\t', index=False)

now we need to process "review" as before and the rest grouped by full phrase

In [ ]:
def study_change(results_df):
    acc = 0
    acc_strict = 0
    n_targets = 0
    not_changed = []
    table_1 = defaultdict(list)
    table_2 = defaultdict(list)
    for target, t_result in results_df.groupby("target"):
        #print(target)
        sp = target.split(' ')
        if (not (len(sp)==2)) and (not (sp[0].lower() in {"the", "an"})) and (target.lower() != "this"):
            year_predictions = []

            has_changed = False
            n_targets += 1
            defs_dict = defaultdict(list)
            for i, (year, y_result) in enumerate(t_result.groupby("year")):
                #print(year)
                defs = set([x.strip() for x in y_result.prediction.unique()])
                for _, sample in y_result.iterrows():
                    pred = sample.prediction.strip()
                    sense_id = defs_dict.get(pred)
                    if sense_id is not None:
                        sense_id = sense_id[0][0]
                    else:
                        sense_id = f'{target.replace(" ", "_")}_{len(defs_dict)+1}'
                    defs_dict[pred].append((sense_id, sample.example, sample.document, year))
                if i != 0:
                    for pred in year_predictions:
                        if pred[1] != defs: # previous year != current

                            has_changed = True
                            table_2["target"].append(target)
                            table_2["years"].append(f"{pred[0]}, {year}")
                            sense_ids_year_1, sense_ids_year_2, senses_year_1, senses_year_2 = [], [], [], []
                            for previous in sorted(list(pred[1])):
                                sense_ids_year_1.extend([x[0] for x in defs_dict[previous]])
                                senses_year_1.append(previous)
                            for current in sorted(list(defs)):
                                sense_ids_year_2.extend([x[0] for x in defs_dict[current]])
                                senses_year_2.append(current)
                            table_2["sense_ids_year_1"].append(",".join(sorted(list(set(sense_ids_year_1)))))
                            table_2["sense_year_1"].append(",".join(senses_year_1))
                            table_2["sense_ids_year_2"].append(",".join(sorted(list(set(sense_ids_year_2)))))
                            table_2["sense_year_2"].append(",".join(senses_year_2))
                year_predictions.append((year, defs))
                #print("---------------------------")
            for sense, sense_data in defs_dict.items():
                #print(sense_data)
                table_1["target"].append(target)
                table_1["sense_id"].append(sense_data[0][0])
                table_1["sense"].append(sense)
                example = []
                fph = set()
                for instance in sense_data:
                    ex = str(instance[1:]) # is originally a tuple
                    example.append(ex)
                    fph.update(full_phrase)
                table_1["examples"].append(", ".join(example))
            if not has_changed:
                not_changed.append(target)

    print(not_changed)
    return pd.DataFrame(table_1), pd.DataFrame(table_2), not_changed

In [ ]:
t1, t2, not_changed = study_change(results_review)

In [ ]:
t1 

In [ ]:
t1.shape

In [ ]:
t2.shape

In [ ]:
t1.to_csv("t1_ours-long-review.tsv", sep="\t", index=False)

In [ ]:
apd = pd.read_csv("apd-xld-eu-long.tsv", sep="\t")
apd.head()

In [ ]:
len(t1.target.unique())

In [ ]:
t2

In [ ]:
merged = pd.merge(t2, apd, on=["target", "years"])

In [ ]:
merged.head()

In [ ]:
merged.sort_values("apd", ascending=False)

In [ ]:
t2_sorted = merged[merged.target.isin(t1.target)].sort_values("apd", ascending=False)

In [ ]:
t2_sorted.to_csv("t2_ours-long-review.tsv", sep="\t", index=False)

In [ ]:
def study_change(results_df):
    acc = 0
    acc_strict = 0
    n_targets = 0
    not_changed = []
    table_1 = defaultdict(list)
    table_2 = defaultdict(list)
    for target, t_result in results_df.groupby("full_phrase"):
        year_predictions = []

        has_changed = False
        n_targets += 1
        defs_dict = defaultdict(list)
        for i, (year, y_result) in enumerate(t_result.groupby("year")):
            defs = set([x.strip() for x in y_result.prediction.unique()])
            for _, sample in y_result.iterrows():
                pred = sample.prediction.strip()
                sense_id = defs_dict.get(pred)
                if sense_id is not None:
                    sense_id = sense_id[0][0]
                else:
                    sense_id = f'{target.replace(" ", "_")}_{len(defs_dict)+1}'
                defs_dict[pred].append((sense_id, sample.example, sample.document, year))
            if i != 0:
                for pred in year_predictions:
                    if pred[1] != defs: # previous year != current

                        has_changed = True
                        table_2["full_phrase"].append(target)
                        actual_targets = y_result.target.unique()
                        
                        table_2["actual_targets"].append(', '.join(actual_targets))
                        
                        years = f"{pred[0]}, {year}"
                        table_2["years"].append(years)
                        apds_list = []
                        for actual_target in actual_targets:
                            apds = apd[apd.target == actual_target]
                            print(actual_target)
                            apds = apds[apds.years == years]
                            
                            apds_list.append(apds.apd.max())
                        table_2["highest_apd"].append(max(apds_list))
                        sense_ids_year_1, sense_ids_year_2, senses_year_1, senses_year_2 = [], [], [], []
                        for previous in sorted(list(pred[1])):
                            sense_ids_year_1.extend([x[0] for x in defs_dict[previous]])
                            senses_year_1.append(previous)
                        for current in sorted(list(defs)):
                            sense_ids_year_2.extend([x[0] for x in defs_dict[current]])
                            senses_year_2.append(current)
                        table_2["sense_ids_year_1"].append(",".join(sorted(list(set(sense_ids_year_1)))))
                        table_2["sense_year_1"].append(",".join(senses_year_1))
                        table_2["sense_ids_year_2"].append(",".join(sorted(list(set(sense_ids_year_2)))))
                        table_2["sense_year_2"].append(",".join(senses_year_2))
            year_predictions.append((year, defs))
            #print("---------------------------")
        for sense, sense_data in defs_dict.items():
            #print(sense_data)
            table_1["target"].append(target)
            table_1["sense_id"].append(sense_data[0][0])
            table_1["sense"].append(sense)
            example = []
            fph = set()
            for instance in sense_data:
                ex = str(instance[1:]) # is originally a tuple
                example.append(ex)
                fph.update(full_phrase)
            table_1["examples"].append(", ".join(example))
        if not has_changed:
            not_changed.append(target)

    print(not_changed)
    return pd.DataFrame(table_1), pd.DataFrame(table_2), not_changed

In [ ]:
t1, t2, not_changed = study_change(results_nice)

In [ ]:
results_nice.full_phrase.unique()

In [ ]:
t1

In [ ]:
t2

In [ ]:
t2.full_phrase.to_list()

In [ ]:
t2.actual_targets.to_list()

In [ ]:
t2_sorted = t2.sort_values("highest_apd", ascending=False)
t2_sorted.to_csv("t2_ours-long-nice.tsv", sep="\t", index=False)
t1.to_csv("t1_ours-long-nice.tsv", sep="\t", index=False)

In [ ]:
t2_sorted